# Portfolio 1.0x Long — REFACTORED
Using modularized `signals.py` and `portfolio_engine.py`

This notebook replaces 5227 lines of monolithic code with ~150 lines of modular imports and execution.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Import modularized components
from signals import _SIGNAL_REGISTRY, _SIGNAL_REGISTRY  # signal functions
from portfolio_engine import (
    load_price_data,
    run_period,
    sharpe,
    cagr,
    max_drawdown,
    perf_stats,
    portfolio_row
)

print('✓ All modules imported successfully')

## Configuration

In [ ]:
# API Configuration
POLYGON_API_KEY = 'REMOVED_API_KEY'  # ⚠️ Move to environment variable

# Period Configuration
TRAIN_START = '2000-01-01'
TRAIN_END = '2015-12-31'
VAL_START = '2016-01-01'
VAL_END = '2021-12-31'
BLIND_START = '2022-01-01'
BLIND_END = '2025-06-30'

# Portfolio Configuration
PORTFOLIO_NAME = '1.0x Long'
LONG_LEVERAGE = 1.0
SHORT_LEVERAGE = 1.0

# ETF Universe from original configuration
ETF_UNIVERSE = [
    'AAXJ', 'ACWI', 'ACWX', 'AGG', 'AGQ', 'AIA', 'AOA', 'AOM', 'AOR', 'BAB',
    'BIL', 'BIV', 'BLV', 'BND', 'BSV', 'BWX', 'CGW', 'CMF', 'CQQQ', 'CWB',
    'CWI', 'DBC', 'DEM', 'DES', 'DGS', 'DHS', 'DIA', 'DLN', 'DLS', 'DON',
    'EDV', 'EEM', 'EFA', 'EFV', 'EFG', 'EPI', 'EPS', 'EWA', 'EWC', 'EWG',
    'EWJ', 'EWL', 'EWP', 'EWS', 'EWT', 'EWU', 'EWW', 'EWY', 'EWZ', 'EXI',
    'EZU', 'FAS', 'FBT', 'FDL', 'FDN', 'FEX', 'FIW', 'FTA', 'FTC', 'FVD',
    'FXH', 'FXI', 'FXL', 'FXO', 'FXR', 'FXU', 'FYX', 'GDX', 'GDXJ', 'GLD',
    'GRID', 'GOVI', 'GSG', 'GSY', 'GVI', 'HYD', 'HYG', 'IAI', 'IAU', 'IBB',
    'ICF', 'ICLN', 'IDU', 'IDV', 'IEF', 'IEI', 'IEV', 'IGF', 'IGM', 'IGV',
    'IHI', 'IJH', 'IJJ', 'IJR', 'IJT', 'IJS', 'IWB', 'IWC', 'IWD', 'IWF',
    'IWL', 'IWM', 'IWN', 'IWO', 'IWP', 'IWR', 'IWS', 'IWV', 'IWX', 'IWY',
    'IXC', 'IXJ', 'IXN', 'IYC', 'IYE', 'IYF', 'IYG', 'IYH', 'IYJ', 'IYK',
    'IYR', 'IYW', 'IYY', 'JNK', 'KBE', 'KRE', 'LQD', 'MDBX', 'MDY', 'MDYG',
    'MDYV', 'MGC', 'MGK', 'MGV', 'MINT', 'MLPS', 'MUB', 'MUNI', 'NYF', 'OEF',
    'ONEQ', 'PCY', 'PDP', 'PEY', 'PFF', 'PGX', 'PHO', 'PID', 'PKW', 'PRF',
    'PRFZ', 'PWB', 'PWV', 'PWZ', 'PXF', 'PXH', 'PZA', 'QQEW', 'QQQ', 'QLD',
    'QTEC', 'ROM', 'RPG', 'RPV', 'RSP', 'RWJ', 'RWK', 'RWL', 'RWO', 'RWR',
    'SCZ', 'SDY', 'SH', 'SHM', 'SHV', 'SHY', 'SIVR', 'SLV', 'SMH', 'SOXX',
    'SPAB', 'SPIB', 'SPIP', 'SPLB', 'SPMB', 'SPSB', 'SPTI', 'SPTL', 'SPMD',
    'SPYG', 'SPYV', 'SCHA', 'SCHB', 'SCHF', 'SCHG', 'SCHV', 'SCHX', 'SLYG',
    'SLYV', 'SPHQ', 'SPXL', 'SUB', 'TBF', 'TECL', 'TFI', 'TIP', 'TLH', 'TLT',
    'TMF', 'TNA', 'UPRO', 'USB', 'USIG', 'USO', 'VAW', 'VB', 'VBK', 'VBR',
    'VCR', 'VCSH', 'VCIT', 'VCLT', 'VDE', 'VDC', 'VEA', 'VEU', 'VFH', 'VGK',
    'VGIT', 'VGLT', 'VGSH', 'VHT', 'VIG', 'VIS', 'VNQ', 'VOE', 'VOT', 'VOX',
    'VPL', 'VPU', 'VTI', 'VTV', 'VUG', 'VV', 'VWO', 'VYM', 'XBI', 'XLE',
    'XLF', 'XLG', 'XLI', 'XLK', 'XLP', 'XLU', 'XLV', 'XLY', 'XME', 'XOP',
    'XSD', 'YINN', 'ZROZ'
]

print(f'Portfolio: {PORTFOLIO_NAME}')
print(f'Period: {TRAIN_START} to {BLIND_END}')
print(f'ETF Universe: {len(ETF_UNIVERSE)} tickers')

## Load Data & Run Backtest

In [ ]:
# Load price data (with caching)
price_data = load_price_data(ETF_UNIVERSE, POLYGON_API_KEY)

# Verify SPY loaded (required)
assert 'SPY' in price_data, 'SPY must be in price data'
print(f"✓ Loaded {len(price_data)} ETFs")

In [ ]:
# Run complete backtest (full period: train + val + blind)
pnl_full, pos_full, ret_full, ind_pnl_full = run_period(
    TRAIN_START,
    BLIND_END,
    price_data,
    _SIGNAL_REGISTRY,
    long_leverage=LONG_LEVERAGE,
    short_leverage=SHORT_LEVERAGE
)

print('✓ Backtest complete')
print(f'  Portfolio Length: {len(pnl_full)} days')
print(f'  Active ETFs: {len(ret_full.columns)}')

## Calculate Metrics

In [ ]:
# Slice periods from full series
train_pnl = pnl_full.loc[TRAIN_START:TRAIN_END]
val_pnl = pnl_full.loc[VAL_START:VAL_END]
blind_pnl = pnl_full.loc[BLIND_START:BLIND_END]
train_val_pnl = pnl_full.loc[TRAIN_START:VAL_END]

# Calculate Sharpe ratios
train_sharpe = sharpe(train_pnl)
val_sharpe = sharpe(val_pnl)
blind_sharpe = sharpe(blind_pnl)
train_val_sharpe = sharpe(train_val_pnl)

print('\n=== PERFORMANCE METRICS ===')
print(f'TRAIN+VAL SHARPE: {train_val_sharpe:.10f}')
print(f'TRAIN SHARPE:     {train_sharpe:.10f}')
print(f'VAL SHARPE:       {val_sharpe:.10f}')
print(f'BLIND SHARPE:     {blind_sharpe:.10f}')

## Equity Curve Visualization

In [ ]:
# Create equity curve
portfolio = (1 + pnl_full).cumprod()

# Plot
plt.figure(figsize=(15, 6))
plt.plot(portfolio.index, portfolio.values, linewidth=2, label=PORTFOLIO_NAME)

# Add period shading
plt.axvspan(pd.to_datetime(TRAIN_START), pd.to_datetime(TRAIN_END),
            color='green', alpha=0.08, label='Train')
plt.axvspan(pd.to_datetime(VAL_START), pd.to_datetime(VAL_END),
            color='blue', alpha=0.08, label='Validation')
plt.axvspan(pd.to_datetime(BLIND_START), pd.to_datetime(BLIND_END),
            color='red', alpha=0.08, label='Blind')

plt.legend()
plt.title('Portfolio Equity Curve — Train / Validation / Blind')
plt.ylabel('Cumulative Return')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Performance Statistics

In [ ]:
# Slice position and return data for each period
train_pos = pos_full.loc[TRAIN_START:TRAIN_END]
val_pos = pos_full.loc[VAL_START:VAL_END]
blind_pos = pos_full.loc[BLIND_START:BLIND_END]

train_ind_pnl = ind_pnl_full.loc[TRAIN_START:TRAIN_END]
val_ind_pnl = ind_pnl_full.loc[VAL_START:VAL_END]
blind_ind_pnl = ind_pnl_full.loc[BLIND_START:BLIND_END]

# Compute per-ETF statistics
train_stats = perf_stats(train_ind_pnl, train_pos, price_data)
val_stats = perf_stats(val_ind_pnl, val_pos, price_data)
blind_stats = perf_stats(blind_ind_pnl, blind_pos, price_data)

# Combine into single table
perf_table = (
    train_stats.add_suffix('_Train')
    .join(val_stats.add_suffix('_Val'))
    .join(blind_stats.add_suffix('_Blind'))
)

# Portfolio row
portfolio_perf = pd.concat([
    portfolio_row(train_pnl, train_pos, price_data, TRAIN_START, TRAIN_END).add_suffix('_Train'),
    portfolio_row(val_pnl, val_pos, price_data, VAL_START, VAL_END).add_suffix('_Val'),
    portfolio_row(blind_pnl, blind_pos, price_data, BLIND_START, BLIND_END).add_suffix('_Blind')
])
perf_table.loc['PORTFOLIO'] = portfolio_perf

print('\nTOP 10 ETFs BY TRAIN SHARPE')
print(perf_table.sort_values('Sharpe_Train', ascending=False).head(10))